In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

DATA = Path.cwd().parent / "data"
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
DEPOT_COLOR = {"Peliyagoda": "#2a78d6", "Kandy": "#eb6834"}
DEFERRED, NOT_RUN = "#2a78d6", "#eb6834"
DELIVERED, DEFERRED_H = "#86b6ef", "#1c5cab"
ALL_ROUTES, DRY, CHILLED = "#52514e", "#2a78d6", "#1baf7a"
BLUES = LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
SURFACE, INK, INK_2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 12, "axes.titlelocation": "left", "axes.titlepad": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK_2, "ytick.labelcolor": INK_2,
    "font.size": 10,
})

In [ ]:
orders = pd.concat([pd.read_csv(DATA / "Training Data/deliveries_train.csv"),
                    pd.read_csv(DATA / "Test Data/task1_test_inputs.csv")], ignore_index=True)
chilled = orders[(orders.brand == "Fresh") & (orders.temp_requirement == "chilled")].copy()
chilled["date"] = pd.to_datetime(chilled.order_date)
chilled["dow"] = chilled.date.dt.dayofweek
outlets = pd.read_csv(DATA / "General Data/outlets.csv").set_index("outlet_id")
vehicles = pd.read_csv(DATA / "General Data/vehicles.csv").rename(columns=str.strip).set_index("vehicle_id")
travel = pd.read_csv(DATA / "General Data/district_travel.csv").set_index("district")
print(f"{len(chilled):,} Fresh chilled orders")

In [ ]:
n_days = chilled.date.nunique()
s = chilled.groupby("district").agg(
    depot=("depot", "first"), outlets=("outlet_id", "nunique"), orders=("delivery_id", "size"),
    volume=("order_volume_m3", "sum"), median_m3=("order_volume_m3", "median"),
    deferred=("dispatch_status", lambda x: (x == "deferred").sum()),
    not_run=("dispatch_status", lambda x: (x == "not_run").sum()))
s["orders_per_day"] = s.orders / n_days
s["m3_per_day"] = s.volume / n_days
s["deferred_rate"] = s.deferred / s.orders
s["not_run_rate"] = s.not_run / s.orders
s = s.join(travel[["depot_to_district_km", "inter_stop_km", "road_class"]])
s.drop(columns="volume").sort_values("deferred_rate", ascending=False).round(3)

In [ ]:
order = s.sort_values("m3_per_day").index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5), sharey=True)
y = np.arange(len(order))
for ax, (col, fmt, title) in zip(axes, [("orders_per_day", "{:.1f}", "Chilled orders per day"),
                                        ("m3_per_day", "{:.1f} m³", "Chilled volume per day"),
                                        ("median_m3", "{:.2f} m³", "Median chilled order size")]):
    vals = s.loc[order, col]
    ax.barh(y, vals, color=[DEPOT_COLOR[d] for d in s.loc[order, "depot"]], height=0.62)
    for yi, v in zip(y, vals):
        ax.text(v, yi, " " + fmt.format(v), va="center", fontsize=8.5, color=INK_2)
    ax.set_yticks(y, order)
    ax.grid(axis="y", visible=False)
    ax.set_title(title)
    ax.margins(x=0.18)
for ax in axes[1:]:
    ax.tick_params(axis="y", length=0)
fig.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in DEPOT_COLOR.values()], [f"{d} depot" for d in DEPOT_COLOR],
           frameon=False, loc="upper left", bbox_to_anchor=(0.07, 0.97), ncol=2, labelcolor=INK_2)
fig.suptitle("Fresh chilled demand by district", x=0.07, ha="left", fontsize=14, color=INK, y=1.03)
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

In [ ]:
order = s.assign(bad=s.deferred_rate + s.not_run_rate).sort_values("bad").index.tolist()
fig, ax = plt.subplots(figsize=(11, 5.5))
y = np.arange(len(order))
d, n = s.loc[order, "deferred_rate"] * 100, s.loc[order, "not_run_rate"] * 100
ax.barh(y, d, color=DEFERRED, height=0.62, label="deferred")
ax.barh(y, n, left=d, color=NOT_RUN, height=0.62, label="not run", edgecolor=SURFACE, linewidth=1.5)
for yi, dist in zip(y, order):
    r = s.loc[dist]
    text = f" {r.deferred_rate:.1%} deferred ({r.deferred:,})"
    if r.not_run:
        text += f" · {r.not_run_rate:.1%} not run ({r.not_run:,})"
    ax.text((r.deferred_rate + r.not_run_rate) * 100, yi, text, va="center", fontsize=8.5, color=INK_2)
ax.set_yticks(y, [f"{dist} ({s.loc[dist, 'depot']})" for dist in order])
ax.xaxis.set_major_formatter(lambda v, _: f"{v:g}%")
ax.set_xlim(0, 45)
ax.grid(axis="y", visible=False)
ax.set_xlabel("% of the district's chilled orders")
ax.legend(frameon=False, loc="lower right", labelcolor=INK_2)
ax.set_title(f"Fresh chilled orders deferred or not run, by district ({s.deferred.sum():,} deferred, {s.not_run.sum():,} not run)")
plt.show()

In [ ]:
deferred = chilled[chilled.dispatch_status == "deferred"]
order = s.sort_values("deferred", ascending=False).index.tolist()
counts = deferred.groupby(["district", "dow"]).size().unstack(fill_value=0).reindex(index=order, columns=range(6), fill_value=0)
totals = chilled.groupby(["district", "dow"]).size().unstack(fill_value=0).reindex(index=order, columns=range(6))
rate = counts / totals
vmax = np.nanmax(rate.values)
cmap = BLUES.copy()
cmap.set_bad(SURFACE)
fig, ax = plt.subplots(figsize=(10, 6.5))
ax.imshow(np.ma.masked_invalid(rate.values), cmap=cmap, aspect="auto", vmin=0, vmax=vmax)
for i in range(len(order)):
    for j in range(6):
        n, r = counts.iat[i, j], rate.iat[i, j]
        text, dark = ("no orders", False) if r != r else ((f"{n:,}\n{r:.1%}" if n else "0"), r > vmax * 0.45)
        ax.text(j, i, text, ha="center", va="center", fontsize=8.5, color=SURFACE if dark else INK_2, linespacing=1.2)
ax.set_xticks(range(6), DAYS)
ax.set_yticks(range(len(order)), order)
ax.tick_params(length=0)
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_title("Deferred Fresh chilled orders by district and weekday (count and share of that day's orders)")
plt.show()

In [ ]:
ran = chilled[chilled.dispatch_status == "attempted"]
routes = ran.groupby("route_id").agg(district=("district", "first"), m3=("order_volume_m3", "sum"),
                                     stops=("delivery_id", "size"), vehicle_id=("vehicle_id", "first"))
routes["util"] = routes.m3 / routes.vehicle_id.map(vehicles.volume_cap_m3)
routes["van"] = routes.vehicle_id.map(vehicles.type) == "van"
order = routes.groupby("district").util.median().sort_values().index.tolist()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True, gridspec_kw=dict(width_ratios=[2, 1]))
ax1.boxplot([routes.loc[routes.district == d, "util"].values * 100 for d in order], orientation="horizontal",
            widths=0.6, showfliers=False, patch_artist=True, boxprops=dict(facecolor=DEFERRED, edgecolor=DEFERRED),
            medianprops=dict(color=SURFACE, linewidth=1.5), whiskerprops=dict(color=MUTED), capprops=dict(color=MUTED))
ax1.set_yticks(range(1, len(order) + 1), order)
ax1.set_xlim(0, 100)
ax1.xaxis.set_major_formatter(lambda v, _: f"{v:g}%")
ax1.set_xlabel("Route volume ÷ vehicle volume capacity")
ax1.grid(axis="y", visible=False)
ax1.set_title("How full chilled routes are")
stops = routes.groupby("district").stops.mean().reindex(order)
vans = routes.groupby("district").van.mean().reindex(order)
y = np.arange(1, len(order) + 1)
ax2.barh(y, stops, color=DEFERRED, height=0.62)
for yi, st, v in zip(y, stops, vans):
    ax2.text(st, yi, f" {st:.1f}" + (f"  ({v:.0%} by van)" if v >= 0.005 else ""), va="center", fontsize=8.5, color=INK_2)
ax2.grid(axis="y", visible=False)
ax2.margins(x=0.45)
ax2.set_xlabel("Stops per route (average)")
ax2.set_title("Stops per chilled route")
ax2.tick_params(axis="y", length=0)
fig.suptitle(f"Fresh chilled routes by district ({len(routes):,} routes)", x=0.07, ha="left", fontsize=14, color=INK, y=1.02)
plt.show()
routes.groupby("district").util.median().sort_values().round(2)

In [ ]:
focus = ["Colombo", "Gampaha"]
g = chilled[chilled.district.isin(focus)].groupby("outlet_id").agg(
    district=("district", "first"), orders=("delivery_id", "size"),
    deferred=("dispatch_status", lambda x: (x == "deferred").mean()),
    not_run=("dispatch_status", lambda x: (x == "not_run").mean()))
g = g.join(outlets[["parking_constraint", "dock_type", "window_open_time", "window_close_time"]])
fig, axes = plt.subplots(2, 1, figsize=(11, 10), sharex=True,
                         gridspec_kw=dict(height_ratios=[g.district.eq(d).sum() for d in focus]))
for ax, dist in zip(axes, focus):
    sub = g[g.district == dist].sort_index(ascending=False)
    y = np.arange(len(sub))
    ax.barh(y, sub.deferred * 100, color=DEFERRED, height=0.65, label="deferred")
    ax.barh(y, sub.not_run * 100, left=sub.deferred * 100, color=NOT_RUN, height=0.65,
            edgecolor=SURFACE, linewidth=1.5, label="not run")
    for yi, r in zip(y, sub.itertuples()):
        ax.text((r.deferred + r.not_run) * 100, yi,
                f" {r.deferred:.0%} deferred" + (f" + {r.not_run:.0%} not run" if r.not_run >= 0.005 else ""),
                va="center", fontsize=8.5, color=INK_2)
    ax.set_yticks(y, [f"{i} · {'VAN ONLY' if p == 'van_only' else d.replace('_', ' ')} · {o}–{cl}"
                      for i, p, d, o, cl in zip(sub.index, sub.parking_constraint, sub.dock_type,
                                                sub.window_open_time, sub.window_close_time)], fontsize=8.5)
    ax.grid(axis="y", visible=False)
    ax.set_xlim(0, 65)
    ax.xaxis.set_major_formatter(lambda v, _: f"{v:g}%")
    ax.set_title(f"{dist} ({len(sub)} outlets)")
axes[1].set_xlabel("% of the outlet's chilled orders")
axes[0].legend(frameon=False, loc="lower right", labelcolor=INK_2)
fig.suptitle("Colombo and Gampaha: chilled orders deferred or not run, by outlet", x=0.07, ha="left", fontsize=14, color=INK, y=1.0)
fig.tight_layout(rect=(0, 0.02, 1, 0.97))
plt.show()
g.sort_values(["district", "deferred"]).round(3)

In [ ]:
status = [("attempted", "delivered", DELIVERED), ("deferred", "deferred", DEFERRED_H), ("not_run", "not run", NOT_RUN)]
for district in focus:
    f = chilled[chilled.district == district]
    for column, unit, label in [("order_volume_m3", "m³", "volume"), ("order_weight_kg", "kg", "weight")]:
        hi = np.percentile(f[column], 99.5)
        bins = np.linspace(0, hi, 36)
        overall = f[column].median()
        dec = 0 if unit == "kg" else 2
        fig, axes = plt.subplots(2, 3, figsize=(15, 7.5), sharex=True, sharey=True)
        rows = []
        for d, ax in enumerate(axes.flat):
            day = f[f.dow == d]
            ax.hist([day.loc[day.dispatch_status == st, column].clip(upper=hi) for st, _, _ in status], bins=bins,
                    stacked=True, color=[c for *_, c in status], label=[n for _, n, _ in status],
                    edgecolor=SURFACE, linewidth=0.5)
            ax.axvline(overall, color=MUTED, lw=1, ls="--")
            med = day[column].median()
            ax.axvline(med, color=INK, lw=1)
            ax.annotate(f"median {med:,.{dec}f} {unit}", (med, 1), xycoords=("data", "axes fraction"),
                        xytext=(4, -2), textcoords="offset points", fontsize=8.5, color=INK, va="top")
            share = (day.dispatch_status != "attempted").mean()
            ax.set_title(f"{DAYS[d]} ({len(day):,} orders, {share:.0%} deferred or not run)", fontsize=11)
            ax.grid(axis="x", visible=False)
            rows.append(dict(day=DAYS[d], orders=len(day), median=round(med, dec + 1),
                             median_delivered=round(day.loc[day.dispatch_status == "attempted", column].median(), dec + 1),
                             median_deferred=round(day.loc[day.dispatch_status == "deferred", column].median(), dec + 1),
                             deferred_or_not_run=f"{share:.0%}"))
        for ax in axes[1]:
            ax.set_xlabel(f"{label.capitalize()} per order ({unit})")
        for ax in axes[:, 0]:
            ax.set_ylabel("Orders")
        axes[0, 2].legend(frameon=False, loc="center right", labelcolor=INK_2, fontsize=9)
        fig.suptitle(f"{district}: Fresh chilled order {label} by day of week ({len(f):,} orders)",
                     x=0.07, ha="left", fontsize=14, color=INK, y=1.0)
        fig.tight_layout(rect=(0, 0, 1, 0.97))
        plt.show()
        print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fresh_ran = orders[(orders.brand == "Fresh") & (orders.dispatch_status == "attempted")]
fresh_routes = fresh_ran.groupby("route_id").agg(district=("district", "first"), temp=("temp_requirement", "first"),
                                                 stops=("outlet_id", "nunique"))
table = fresh_routes.groupby(["district", "temp"]).stops.mean().unstack()
table["all"] = fresh_routes.groupby("district").stops.mean()
table["routes"] = fresh_routes.groupby("district").size()
table["outlets"] = fresh_ran.groupby("district").outlet_id.nunique()
table = table.sort_values("all")

fig, ax = plt.subplots(figsize=(11, 8))
y = np.arange(len(table))
for k, (col, label, color) in enumerate([("all", "all Fresh routes", ALL_ROUTES), ("ambient", "dry (ambient) routes", DRY),
                                         ("chilled", "chilled routes", CHILLED)]):
    pos = y + (1 - k) * 0.26
    ax.barh(pos, table[col], height=0.26, color=color, label=label)
    for p, v in zip(pos, table[col]):
        ax.text(v, p, f" {v:.1f}", va="center", fontsize=8, color=INK_2)
ax.set_yticks(y, [f"{d} ({int(r.outlets)} outlets)" for d, r in table.iterrows()])
ax.grid(axis="y", visible=False)
ax.set_xlabel("Stops per route (average)")
ax.margins(x=0.08)
ax.legend(frameon=False, loc="lower right", labelcolor=INK_2)
ax.set_title(f"Fresh stops per route by district ({len(fresh_routes):,} routes)")
plt.show()
table.round(2)